# VLegalAI RAG / GraphRAG evaluation with RAGAS

Notebook này chạy đánh giá bằng DeepSeek local qua Hugging Face.

- Không đọc env.json và không dùng API key cho LLM.
- Kaggle: project/dataset/model được đọc từ /kaggle/input; kết quả ghi vào /kaggle/working.
- DEEPSEEK_MODEL_PATH có thể đặt qua biến môi trường hoặc chỉnh trong cell cấu hình.
- Model được load bằng device_map="balanced" trên hai GPU trong DEEPSEEK_GPU_IDS.
- Batch mặc định trên Kaggle là RAGAS 8, DeepSeek pipeline 8, embedding 64; có thể tăng qua env.

In [ ]:
# Chỉ cài package còn thiếu; sau khi restart kernel, cell này sẽ tự bỏ qua pip.
import importlib.util
import subprocess
import sys

required_packages = {
    'ragas': ('ragas', 'ragas==0.2.15'),
    'dotenv': ('python-dotenv', 'python-dotenv'),
    'transformers': ('transformers', 'transformers'),
    'accelerate': ('accelerate', 'accelerate'),
    'sentencepiece': ('sentencepiece', 'sentencepiece'),
    'sentence_transformers': ('sentence-transformers', 'sentence-transformers'),
    'langchain_huggingface': ('langchain-huggingface', 'langchain-huggingface'),
    'huggingface_hub': ('huggingface-hub', 'huggingface-hub'),
}
missing_packages = [
    package_spec
    for module_name, (package_name, package_spec) in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--disable-pip-version-check',
        '--upgrade-strategy', 'only-if-needed',
        *missing_packages,
    ])
    print('Package installation completed. Restart the kernel once, then run from the top.')
else:
    print('All required packages are already installed; skip pip install.')

In [ ]:
from __future__ import annotations

import copy
import csv
import json
import math
import os
import hashlib
import re
import shutil
import sqlite3
import unicodedata
import socket
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
from array import array
from collections import OrderedDict

import pandas as pd
import requests
from dotenv import load_dotenv

IS_KAGGLE = Path("/kaggle").exists()
KAGGLE_INPUT_DIR = Path(os.getenv("KAGGLE_INPUT_DIR", "/kaggle/input"))
KAGGLE_WORKING_DIR = Path(os.getenv("KAGGLE_WORKING_DIR", "/kaggle/working"))

# Kaggle input chỉ có dữ liệu/SQLite, notebook không yêu cầu app/main.py.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DOTENV_PATH = PROJECT_ROOT / ".env"
if DOTENV_PATH.exists():
    load_dotenv(DOTENV_PATH, override=True, interpolate=True)

# Bắt buộc Hugging Face/Transformers chỉ dùng file local, không gọi model hub.
# Cho phép tải model ở cell cấu hình; sau khi tải xong sẽ chuyển sang offline.
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

VECTOR_DIMS = 384
VN_WORD_RE = re.compile(r'[0-9A-Za-zÀ-ỹĐđ]+', re.UNICODE)

def repair_text(value: Any) -> str:
    if value is None:
        return ''
    current = str(value)
    for _ in range(2):
        try:
            repaired = current.encode('latin1').decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            break
        if repaired == current:
            break
        current = repaired
    return current

def strip_accents(text: str) -> str:
    text = text.replace('Đ', 'D').replace('đ', 'd')
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

def words_for_vector(text: str) -> list[str]:
    words = [word for word in VN_WORD_RE.findall(strip_accents(text).lower()) if len(word) >= 2]
    grams: list[str] = []
    for word in words:
        grams.append(f'w:{word}')
        if len(word) >= 5:
            grams.extend(f'c:{word[i:i + 4]}' for i in range(len(word) - 3))
    return grams

def hash_vector(text: str, dims: int = VECTOR_DIMS) -> array:
    vector = array('f', [0.0] * dims)
    for gram in words_for_vector(text):
        digest = hashlib.blake2b(gram.encode('utf-8'), digest_size=8).digest()
        index = int.from_bytes(digest[:4], 'little') % dims
        vector[index] += (1.3 if gram.startswith('w:') else 0.45) * (1.0 if digest[4] & 1 else -1.0)
    norm = math.sqrt(sum(value * value for value in vector))
    if norm:
        for index, value in enumerate(vector):
            vector[index] = value / norm
    return vector

store_cache: dict[str, Any] = {}
store_backend_errors: dict[str, str] = {}

def normalize_backend(value: str | None) -> str:
    backend = (value or os.getenv('RETRIEVER_BACKEND', 'sqlite')).strip().lower().replace('-', '_')
    return {'local': 'sqlite', 'local_graphrag': 'sqlite', 'sqlite_rag': 'sqlite', 'rag': 'qdrant', 'graphrag': 'neo4j'}.get(backend, backend)

def backend_mode_label(backend: str) -> str:
    return {'sqlite': 'Local GraphRAG (SQLite)', 'qdrant': 'RAG (Qdrant)', 'neo4j': 'GraphRAG (Neo4j)'}.get(backend, backend)

def get_store(backend_override: str | None = None) -> Any:
    canonical = normalize_backend(backend_override)
    if canonical != 'sqlite':
        raise RuntimeError(f'Backend {canonical} không có dữ liệu/service trong Kaggle input; dùng sqlite.')
    if canonical not in store_cache:
        store_cache[canonical] = SQLiteGraphRAGStore(SQLITE_DB_PATH)
    return store_cache[canonical]

if IS_KAGGLE:
    # .env của project có thể chứa đường dẫn Windows; ưu tiên đường dẫn Linux trong Kaggle.
    data_candidates = [
        PROJECT_ROOT / "Data (1)",
        PROJECT_ROOT / "data",
        KAGGLE_INPUT_DIR / "Data (1)",
    ]
    data_dir = next((path for path in data_candidates if path.exists()), None)
    if data_dir:
        os.environ["LEGAL_DATA_DIR"] = str(data_dir)
    storage_candidates = [
        PROJECT_ROOT / "storage" / "graphrag",
        KAGGLE_INPUT_DIR / "storage" / "graphrag",
    ]
    storage_dir = next((path for path in storage_candidates if path.exists()), None)
    if storage_dir:
        os.environ["LEGAL_STORAGE_DIR"] = str(storage_dir)
        db_path = storage_dir / "legal_graphrag.sqlite"
        if db_path.exists():
            os.environ["LEGAL_GRAPHRAG_DB"] = str(db_path)

store_cache.clear()
store_backend_errors.clear()

def normalize_qdrant_url_from_env() -> None:
    raw = (os.getenv("QDRANT_URL") or "").strip().strip('"').strip("'")
    if not raw:
        return
    raw = raw.rstrip("/")
    for marker in ("/dashboard", "/collections", "/collection", "/points"):
        if marker in raw:
            raw = raw.split(marker, 1)[0]
    os.environ["QDRANT_URL"] = raw

normalize_qdrant_url_from_env()

print("Kaggle runtime:", IS_KAGGLE)
print("Kaggle input dir:", KAGGLE_INPUT_DIR)
print("Kaggle working dir:", KAGGLE_WORKING_DIR)
print("Project root:", PROJECT_ROOT)
print("Loaded .env for retrieval configuration:", DOTENV_PATH.exists())
print("RETRIEVER_BACKEND:", os.getenv("RETRIEVER_BACKEND", "auto"))
print("QDRANT_URL:", os.getenv("QDRANT_URL", "unset"))
print("QDRANT_COLLECTION:", os.getenv("QDRANT_COLLECTION", "unset"))
print("NEO4J_URI:", os.getenv("NEO4J_URI", "unset"))
print("Offline Transformers:", os.environ["TRANSFORMERS_OFFLINE"])

In [ ]:
def env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_int(name: str, default: int | None) -> int | None:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    if value.strip().lower() in {"none", "null", "all"}:
        return None
    return int(value)


def env_float(name: str, default: float) -> float:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    try:
        return float(value)
    except ValueError:
        return default


def env_list(name: str, default: list[str]) -> list[str]:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    return [item.strip() for item in value.replace(",", " ").split() if item.strip()]


def first_existing_path(candidates: list[Path], fallback: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return fallback

dataset_name = "eval_legal_rag_graphrag_1000.json"
dataset_candidates = [
    PROJECT_ROOT / dataset_name,
    KAGGLE_INPUT_DIR / dataset_name,
]
if KAGGLE_INPUT_DIR.exists():
    dataset_candidates.extend(KAGGLE_INPUT_DIR.glob(f"**/{dataset_name}"))
DATASET_PATH = Path(os.getenv("RAGAS_EVAL_DATASET", "")) if os.getenv("RAGAS_EVAL_DATASET") else first_existing_path(
    dataset_candidates,
    PROJECT_ROOT / dataset_name,
)
db_name = 'legal_graphrag.sqlite'
db_candidates = [
    PROJECT_ROOT / db_name,
    PROJECT_ROOT / 'storage' / 'graphrag' / db_name,
    KAGGLE_INPUT_DIR / db_name,
]
if KAGGLE_INPUT_DIR.exists():
    db_candidates.extend(KAGGLE_INPUT_DIR.glob(f'**/{db_name}'))
configured_db_path = os.getenv('LEGAL_GRAPHRAG_DB')
SQLITE_DB_PATH = Path(configured_db_path).expanduser() if configured_db_path and Path(configured_db_path).exists() else first_existing_path(db_candidates, KAGGLE_INPUT_DIR / db_name)
os.environ['LEGAL_GRAPHRAG_DB'] = str(SQLITE_DB_PATH)
print('SQLite database:', SQLITE_DB_PATH)
OUT_DIR = Path(
    os.getenv(
        "RAGAS_EVAL_OUT_DIR",
        str(KAGGLE_WORKING_DIR / "ragas_notebook" if IS_KAGGLE else PROJECT_ROOT / "storage" / "eval" / "ragas_notebook"),
    )
)
# Kaggle mặc định dùng SQLite local vì không có sẵn Neo4j/Qdrant service.
DEFAULT_BACKENDS = ["sqlite"] if IS_KAGGLE else ["rag", "graphrag"]
BACKENDS = env_list("RAGAS_EVAL_BACKENDS", DEFAULT_BACKENDS)

TOP_K = env_int("RAGAS_EVAL_TOP_K", 10) or 10
OFFSET = env_int("RAGAS_EVAL_OFFSET", 0) or 0
LIMIT = env_int("RAGAS_EVAL_LIMIT", 1000)

# generate luôn gọi DeepSeek local; dataset/reference dùng dữ liệu có sẵn.
RESPONSE_MODE = os.getenv("RAGAS_RESPONSE_MODE", "generate").strip().lower()
RUN_RAGAS = env_bool("RAGAS_RUN", True)

# Đường dẫn model local. Trên Kaggle nên upload model thành Kaggle Dataset.
model_name = "DeepSeek-R1-Distill-Qwen-7B"
model_candidates = [
    PROJECT_ROOT / "models" / model_name,
    KAGGLE_INPUT_DIR / "deepseek-r1-distill-qwen-7b",
    KAGGLE_INPUT_DIR / "DeepSeek-R1-Distill-Qwen-7B",
]
if KAGGLE_INPUT_DIR.exists():
    model_candidates.extend(
        path
        for path in KAGGLE_INPUT_DIR.glob("**/*DeepSeek*")
        if path.is_dir()
    )
configured_model_path = os.getenv("DEEPSEEK_MODEL_PATH")
DEEPSEEK_MODEL_PATH = Path(configured_model_path).expanduser() if configured_model_path else first_existing_path(
    [path for path in model_candidates if (path / "config.json").exists()],
    model_candidates[0],
)

# Chọn đúng hai GPU trước khi import torch/transformers.
DEEPSEEK_GPU_IDS = [int(value) for value in env_list("DEEPSEEK_GPU_IDS", ["0", "1"])]
if len(DEEPSEEK_GPU_IDS) != 2:
    raise ValueError("DEEPSEEK_GPU_IDS phải chứa đúng hai GPU, ví dụ: 0,1")
if "torch" not in sys.modules:
    os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(value) for value in DEEPSEEK_GPU_IDS)
else:
    print("torch đã được import; CUDA_VISIBLE_DEVICES phải được cấu hình từ kernel Kaggle.")

DEFAULT_BATCH_SIZE = 8 if IS_KAGGLE else 16
DEEPSEEK_GPU_MEMORY_FRACTION = env_float("DEEPSEEK_GPU_MEMORY_FRACTION", 0.82 if IS_KAGGLE else 0.90)
DEEPSEEK_CPU_MEMORY_GB = env_int("DEEPSEEK_CPU_MEMORY_GB", 24 if IS_KAGGLE else 64) or (24 if IS_KAGGLE else 64)
DEEPSEEK_PIPELINE_BATCH_SIZE = env_int("DEEPSEEK_PIPELINE_BATCH_SIZE", DEFAULT_BATCH_SIZE) or DEFAULT_BATCH_SIZE
DEEPSEEK_MAX_NEW_TOKENS = env_int("DEEPSEEK_MAX_NEW_TOKENS", 512 if IS_KAGGLE else 768) or (512 if IS_KAGGLE else 768)
DEEPSEEK_TRUST_REMOTE_CODE = env_bool("DEEPSEEK_TRUST_REMOTE_CODE", False)
DEEPSEEK_MODEL_ID = os.getenv("DEEPSEEK_MODEL_ID", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")
DOWNLOAD_DEEPSEEK_MODEL = env_bool("DOWNLOAD_DEEPSEEK_MODEL", IS_KAGGLE)

def ensure_deepseek_model() -> None:
    global DEEPSEEK_MODEL_PATH
    if (DEEPSEEK_MODEL_PATH / "config.json").is_file():
        print("DeepSeek model found:", DEEPSEEK_MODEL_PATH)
        return
    if not DOWNLOAD_DEEPSEEK_MODEL:
        raise FileNotFoundError(
            f"Không tìm thấy model local: {DEEPSEEK_MODEL_PATH}. "
            "Bật DOWNLOAD_DEEPSEEK_MODEL=true hoặc upload model vào Kaggle Dataset."
        )
    if IS_KAGGLE and str(DEEPSEEK_MODEL_PATH.resolve()).startswith(str(KAGGLE_INPUT_DIR.resolve()) + os.sep):
        DEEPSEEK_MODEL_PATH = KAGGLE_WORKING_DIR / "models" / model_name
    DEEPSEEK_MODEL_PATH.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {DEEPSEEK_MODEL_ID} to {DEEPSEEK_MODEL_PATH} ...")
    previous_hf_offline = os.environ.get("HF_HUB_OFFLINE")
    previous_transformers_offline = os.environ.get("TRANSFORMERS_OFFLINE")
    os.environ["HF_HUB_OFFLINE"] = "0"
    os.environ["TRANSFORMERS_OFFLINE"] = "0"
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id=DEEPSEEK_MODEL_ID,
            local_dir=str(DEEPSEEK_MODEL_PATH),
        )
    except Exception as exc:
        raise RuntimeError(
            "Không tải được DeepSeek. Hãy bật Internet trên Kaggle hoặc upload model local."
        ) from exc
    finally:
        os.environ["HF_HUB_OFFLINE"] = "1"
        os.environ["TRANSFORMERS_OFFLINE"] = "1"
    if not (DEEPSEEK_MODEL_PATH / "config.json").is_file():
        raise FileNotFoundError(f"Download completed but config.json is missing: {DEEPSEEK_MODEL_PATH}")
    print("DeepSeek download completed.")

ensure_deepseek_model()
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# RAGAS batch lớn hơn bản cũ; giảm nếu hết VRAM.
RAGAS_BATCH_SIZE = env_int("RAGAS_BATCH_SIZE", DEFAULT_BATCH_SIZE) or DEFAULT_BATCH_SIZE
RAGAS_EMBEDDING_BATCH_SIZE = env_int("RAGAS_EMBEDDING_BATCH_SIZE", 64) or 64
RAISE_EXCEPTIONS = env_bool("RAGAS_RAISE_EXCEPTIONS", False)

# legal_hash chạy hoàn toàn local và không cần model/API key thứ hai.
LOCAL_EMBEDDING_PROVIDERS = {"huggingface", "hf", "sentence_transformers", "sentence-transformers"}
EMBEDDING_PROVIDER = (os.getenv("RAGAS_EMBEDDING_PROVIDER") or "legal_hash").strip().lower()
EMBEDDING_MODEL = os.getenv("RAGAS_EMBEDDING_MODEL") or (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    if EMBEDDING_PROVIDER in LOCAL_EMBEDDING_PROVIDERS
    else None
)

METRIC_NAMES = env_list(
    "RAGAS_METRICS",
    [
        "context_precision",
        "context_recall",
        "faithfulness",
        "answer_relevancy",
        "answer_correctness",
    ],
)

RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_DIR = OUT_DIR / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("BACKENDS:", BACKENDS)
print("DATASET_PATH:", DATASET_PATH)
print("TOP_K/OFFSET/LIMIT:", TOP_K, OFFSET, LIMIT)
print("RESPONSE_MODE:", RESPONSE_MODE)
print("RUN_RAGAS:", RUN_RAGAS)
print("DEEPSEEK_MODEL_PATH:", DEEPSEEK_MODEL_PATH)
print("DEEPSEEK_MODEL_ID:", DEEPSEEK_MODEL_ID)
print("DOWNLOAD_DEEPSEEK_MODEL:", DOWNLOAD_DEEPSEEK_MODEL)
print("DEEPSEEK_GPU_IDS:", DEEPSEEK_GPU_IDS)
print("DEEPSEEK_PIPELINE_BATCH_SIZE:", DEEPSEEK_PIPELINE_BATCH_SIZE)
print("RAGAS_BATCH_SIZE:", RAGAS_BATCH_SIZE)
print("RAGAS_EMBEDDING_BATCH_SIZE:", RAGAS_EMBEDDING_BATCH_SIZE)
print("EMBEDDING_PROVIDER:", EMBEDDING_PROVIDER)
print("METRIC_NAMES:", METRIC_NAMES)
print("RUN_DIR:", RUN_DIR)

In [ ]:
class SQLiteGraphRAGStore:
    def __init__(self, db_path: Path):
        if not db_path.exists():
            raise FileNotFoundError(f'Không tìm thấy SQLite database: {db_path}')
        if IS_KAGGLE and str(db_path.resolve()).startswith(str(KAGGLE_INPUT_DIR.resolve()) + os.sep):
            runtime_db_path = KAGGLE_WORKING_DIR / 'legal_graphrag_runtime.sqlite'
            if not runtime_db_path.exists() or runtime_db_path.stat().st_size != db_path.stat().st_size:
                print(f'Copying SQLite database to writable path: {runtime_db_path}')
                shutil.copy2(db_path, runtime_db_path)
            db_path = runtime_db_path
        self.db_path = db_path
        self.conn = sqlite3.connect(str(db_path), check_same_thread=False)
        self.conn.row_factory = sqlite3.Row
        self._vectors: list[tuple[str, bytes]] | None = None

    def _fts_query(self, query: str) -> str:
        tokens = [token for token in VN_WORD_RE.findall(query.lower()) if len(token) >= 2]
        stop = {'theo', 'quy', 'dinh', 'cho', 'toi', 'hoi', 'nhu', 'nao', 've', 'va', 'la', 'cua', 'duoc', 'khong', 'trong', 'cac'}
        tokens = [token for token in tokens if strip_accents(token) not in stop]
        tokens = tokens[:18] or VN_WORD_RE.findall(query.lower())[:12]
        return ' OR '.join(f'"{token}"' for token in tokens)

    def _fts_search(self, query: str, limit: int) -> list[sqlite3.Row]:
        expression = self._fts_query(query)
        if not expression:
            return []
        try:
            return self.conn.execute(
                'SELECT c.*, bm25(chunk_fts) AS rank FROM chunk_fts JOIN chunks c ON c.chunk_id = chunk_fts.chunk_id WHERE chunk_fts MATCH ? ORDER BY rank LIMIT ?',
                (expression, limit),
            ).fetchall()
        except sqlite3.OperationalError:
            like = f'%{query[:80]}%'
            return self.conn.execute('SELECT * FROM chunks WHERE text LIKE ? OR title LIKE ? LIMIT ?', (like, like, limit)).fetchall()

    def _load_vectors(self) -> list[tuple[str, bytes]]:
        if self._vectors is None:
            self._vectors = [(row['chunk_id'], row['vector']) for row in self.conn.execute('SELECT chunk_id, vector FROM chunks').fetchall() if row['vector']]
        return self._vectors

    def _vector_search(self, query: str, limit: int) -> list[tuple[str, float]]:
        query_vector = hash_vector(query)
        scored = []
        for chunk_id, blob in self._load_vectors():
            vector = array('f')
            vector.frombytes(blob)
            scored.append((chunk_id, sum(a * b for a, b in zip(query_vector, vector))))
        scored.sort(key=lambda item: item[1], reverse=True)
        return scored[:limit]

    def _chunks_by_ids(self, chunk_ids: list[str]) -> dict[str, sqlite3.Row]:
        ids = list(dict.fromkeys(chunk_ids))
        if not ids:
            return {}
        placeholders = ','.join('?' for _ in ids)
        rows = self.conn.execute(f'SELECT * FROM chunks WHERE chunk_id IN ({placeholders})', ids).fetchall()
        return {row['chunk_id']: row for row in rows}

    def _best_chunk_for_node(self, node_id: str) -> sqlite3.Row | None:
        priority = {'point': 0, 'clause': 1, 'article': 2, 'sliding': 3, 'document_intro': 4, 'structure': 5, 'semantic': 6}
        rows = self.conn.execute('SELECT * FROM chunks WHERE node_id = ?', (node_id,)).fetchall()
        return min(rows, key=lambda row: (priority.get(row['chunk_type'], 99), row['ordinal'])) if rows else None

    def _ancestors(self, node_id: str) -> list[str]:
        ancestors = []
        cursor = node_id
        seen = set()
        while cursor and cursor not in seen and len(ancestors) < 6:
            seen.add(cursor)
            row = self.conn.execute('SELECT parent_id FROM nodes WHERE node_id = ?', (cursor,)).fetchone()
            if not row or not row['parent_id']:
                break
            cursor = row['parent_id']
            ancestors.append(cursor)
        return ancestors

    def retrieve(self, query: str, top_k: int = 10) -> list[dict[str, Any]]:
        query = repair_text(query).strip()
        if not query:
            return []
        combined: dict[str, float] = {}
        reasons: dict[str, list[str]] = {}
        for rank, row in enumerate(self._fts_search(query, max(24, top_k * 4)), start=1):
            combined[row['chunk_id']] = combined.get(row['chunk_id'], 0.0) + 1.25 / (rank + 2)
            reasons.setdefault(row['chunk_id'], []).append('FTS')
        for rank, (chunk_id, score) in enumerate(self._vector_search(query, max(24, top_k * 4)), start=1):
            combined[chunk_id] = combined.get(chunk_id, 0.0) + max(score, 0.0) / math.sqrt(rank + 1)
            reasons.setdefault(chunk_id, []).append('vector')
        rows_by_id = self._chunks_by_ids(list(combined))
        query_ascii = strip_accents(query).lower()
        terms = [term for term in VN_WORD_RE.findall(query_ascii) if len(term) >= 2]
        article_numbers = {match.group(1).lower() for match in re.finditer(r'điều\s+(\d+[a-zA-Z]?)', query_ascii, re.I)}
        clause_numbers = {match.group(1) for match in re.finditer(r'khoản\s+(\d{1,3})', query_ascii, re.I)}
        for chunk_id, row in rows_by_id.items():
            haystack_raw = f"{row['title']} {row['citation']} {row['text'][:800]}"
            haystack = haystack_raw.lower()
            haystack_ascii = strip_accents(haystack_raw).lower()
            if terms:
                coverage = sum(term in haystack_ascii for term in terms) / min(len(terms), 12)
                combined[chunk_id] += coverage * 0.9
                if coverage < 0.18:
                    combined[chunk_id] *= 0.45
            for number in article_numbers:
                if re.search(rf'điều\s+{re.escape(number)}\b', haystack_ascii, re.I):
                    combined[chunk_id] += 0.75
                    reasons.setdefault(chunk_id, []).append('exact-article')
            for number in clause_numbers:
                if re.search(rf'khoản\s+{re.escape(number)}\b', haystack_ascii, re.I):
                    combined[chunk_id] += 0.45
                    reasons.setdefault(chunk_id, []).append('exact-clause')
        base_ids = [chunk_id for chunk_id, _ in sorted(combined.items(), key=lambda item: item[1], reverse=True)[:max(top_k * 2, 12)]]
        expanded: dict[str, dict[str, Any]] = {}
        def add(row: sqlite3.Row, score: float, reason: str) -> None:
            payload = expanded.get(row['chunk_id'])
            if payload is None or score > payload['score']:
                expanded[row['chunk_id']] = {**dict(row), 'score': score, 'reasons': list(dict.fromkeys(reasons.get(row['chunk_id'], []) + [reason]))}
        for chunk_id in base_ids:
            row = rows_by_id.get(chunk_id)
            if not row:
                continue
            base_score = combined[chunk_id]
            add(row, base_score, 'base')
            for position, ancestor_id in enumerate(self._ancestors(row['node_id']), start=1):
                ancestor_chunk = self._best_chunk_for_node(ancestor_id)
                if ancestor_chunk:
                    add(ancestor_chunk, base_score * max(0.28, 0.72 - position * 0.1), 'ancestor')
        selected = sorted(expanded.values(), key=lambda item: item['score'], reverse=True)[:top_k]
        for index, item in enumerate(selected, start=1):
            item['source_id'] = f'S{index}'
            item['score'] = round(float(item['score']), 4)
        return selected

from langchain_core.embeddings import Embeddings


CONTEXT_METRICS = {"context_precision", "context_recall"}
RESPONSE_METRICS = {"faithfulness", "answer_relevancy", "answer_correctness"}


class LegalHashEmbeddings(Embeddings):
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        vector = hash_vector(text or "", dims=VECTOR_DIMS)
        return [float(value) for value in vector]


DEEPSEEK_LLM_CACHE: Any = None


def _deepseek_max_memory(torch: Any) -> dict[Any, str]:
    max_memory: dict[Any, str] = {}
    for visible_index in range(2):
        total_gb = torch.cuda.get_device_properties(visible_index).total_memory / (1024**3)
        usable_gb = max(1, int(total_gb * DEEPSEEK_GPU_MEMORY_FRACTION))
        max_memory[visible_index] = f"{usable_gb}GiB"
    max_memory["cpu"] = f"{DEEPSEEK_CPU_MEMORY_GB}GiB"
    return max_memory


def build_local_deepseek_llm() -> Any:
    global DEEPSEEK_LLM_CACHE
    if DEEPSEEK_LLM_CACHE is not None:
        return DEEPSEEK_LLM_CACHE

    if not DEEPSEEK_MODEL_PATH.is_dir():
        raise FileNotFoundError(
            f"Không tìm thấy model DeepSeek local: {DEEPSEEK_MODEL_PATH}. "
            "Hãy đặt DEEPSEEK_MODEL_PATH tới thư mục model đã tải sẵn."
        )

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError(
            "Cần ít nhất hai GPU CUDA khả dụng. "
            f"Số GPU nhìn thấy hiện tại: {torch.cuda.device_count()}."
        )

    try:
        use_bfloat16 = all(
            torch.cuda.is_bf16_supported(device=index)
            for index in range(2)
        )
    except (AttributeError, TypeError):
        use_bfloat16 = False
    dtype = torch.bfloat16 if use_bfloat16 else torch.float16

    tokenizer = AutoTokenizer.from_pretrained(
        str(DEEPSEEK_MODEL_PATH),
        local_files_only=True,
        trust_remote_code=DEEPSEEK_TRUST_REMOTE_CODE,
        use_fast=True,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        str(DEEPSEEK_MODEL_PATH),
        local_files_only=True,
        trust_remote_code=DEEPSEEK_TRUST_REMOTE_CODE,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        device_map="balanced",
        max_memory=_deepseek_max_memory(torch),
    )
    model.eval()

    device_map = getattr(model, "hf_device_map", {})
    placements = {
        int(value)
        for value in device_map.values()
        if isinstance(value, int) or str(value).isdigit()
    }
    if not {0, 1}.issubset(placements):
        raise RuntimeError(
            "DeepSeek chưa được phân bổ trên cả hai GPU. "
            f"hf_device_map={device_map}"
        )

    text_generator = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        batch_size=DEEPSEEK_PIPELINE_BATCH_SIZE,
        max_new_tokens=DEEPSEEK_MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    hf_llm = HuggingFacePipeline(
        pipeline=text_generator,
        model_id=str(DEEPSEEK_MODEL_PATH),
        batch_size=DEEPSEEK_PIPELINE_BATCH_SIZE,
    )
    DEEPSEEK_LLM_CACHE = ChatHuggingFace(
        llm=hf_llm,
        tokenizer=tokenizer,
        model_id=str(DEEPSEEK_MODEL_PATH),
    )

    print("DeepSeek device map:", device_map)
    print("DeepSeek dtype:", dtype)
    print("DeepSeek GPUs: visible cuda:0 and cuda:1")
    print("DeepSeek batch size:", DEEPSEEK_PIPELINE_BATCH_SIZE)
    return DEEPSEEK_LLM_CACHE


def build_chat_llm(provider: str = "deepseek_local", model: str | None = None) -> Any:
    provider = provider.lower().strip()
    if provider not in {"deepseek", "deepseek_local", "local"}:
        raise ValueError("Notebook này chỉ hỗ trợ provider deepseek_local offline")
    return build_local_deepseek_llm()


def get_answer_llm() -> Any:
    return build_local_deepseek_llm()


def clean_text(value: Any) -> str:
    return repair_text(value).strip()


def backend_slug(value: str) -> str:
    return value.strip().lower().replace("-", "_").replace(" ", "_")


def load_eval_samples(path: Path, offset: int = 0, limit: int | None = None) -> list[dict[str, Any]]:
    rows = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(rows, list):
        raise ValueError(f"Expected a JSON list in {path}")
    end = None if limit is None else offset + max(limit, 0)
    return rows[offset:end]


def question_from_sample(sample: dict[str, Any]) -> str:
    ragas_fields = sample.get("ragas_fields") or {}
    return clean_text(ragas_fields.get("user_input") or sample.get("question") or "")


def reference_from_sample(sample: dict[str, Any]) -> str:
    ragas_fields = sample.get("ragas_fields") or {}
    return clean_text(ragas_fields.get("reference") or sample.get("reference_answer") or "")


def expected_chunk_ids(sample: dict[str, Any]) -> list[str]:
    contexts = sample.get("relevant_contexts") or []
    return [
        clean_text(item.get("chunk_id"))
        for item in contexts
        if isinstance(item, dict) and item.get("chunk_id")
    ]


def context_text_from_source(source: dict[str, Any]) -> str:
    citation = clean_text(source.get("citation") or source.get("title") or "")
    text = clean_text(source.get("text") or "")
    if citation and text:
        return f"{citation}\n{text}"
    return text or citation


def compact_source(source: dict[str, Any]) -> dict[str, Any]:
    return {
        "source_id": source.get("source_id"),
        "chunk_id": source.get("chunk_id"),
        "doc_id": source.get("doc_id"),
        "node_id": source.get("node_id"),
        "chunk_type": clean_text(source.get("chunk_type") or ""),
        "citation": clean_text(source.get("citation") or source.get("title") or ""),
        "score": source.get("score"),
        "reasons": source.get("reasons") or [],
    }


def exact_retrieval_scores(
    retrieved_sources: list[dict[str, Any]],
    expected_ids: list[str],
) -> dict[str, Any]:
    retrieved_ids = [
        clean_text(source.get("chunk_id"))
        for source in retrieved_sources
        if source.get("chunk_id")
    ]
    expected = list(dict.fromkeys(expected_ids))
    retrieved = list(dict.fromkeys(retrieved_ids))
    expected_set = set(expected)
    hits = expected_set & set(retrieved)
    first_rank = next(
        (index for index, chunk_id in enumerate(retrieved_ids, start=1) if chunk_id in expected_set),
        None,
    )
    return {
        "exact_hit_at_k": bool(hits),
        "exact_hits": len(hits),
        "exact_precision": len(hits) / len(retrieved) if retrieved else 0.0,
        "exact_recall": len(hits) / len(expected) if expected else None,
        "exact_mrr": 1.0 / first_rank if first_rank else 0.0,
    }


def build_answer_context(sources: list[dict[str, Any]], max_chars: int = 14000) -> str:
    blocks: list[str] = []
    used = 0
    for index, source in enumerate(sources, start=1):
        citation = clean_text(source.get("citation") or source.get("title") or f"S{index}")
        text = clean_text(source.get("text") or "")
        block = f"[S{index}] {citation}\n{text[:1800]}"
        if used + len(block) > max_chars:
            break
        blocks.append(block)
        used += len(block)
    return "\n\n".join(blocks)


def generate_answer_with_deepseek(
    question: str,
    sources: list[dict[str, Any]],
    mode_label: str,
) -> str:
    if not sources:
        return (
            f"Chưa tìm thấy căn cứ đủ rõ trong chế độ {mode_label} "
            "để trả lời chắc chắn."
        )

    from langchain_core.messages import HumanMessage, SystemMessage

    system_prompt = (
        "Bạn là VLegal AI, trợ lý pháp lý Việt Nam. "
        "Chỉ sử dụng CONTEXT được cung cấp, không bịa căn cứ. "
        "Trả lời ngắn gọn, chính xác, bằng tiếng Việt và trích dẫn [S1], [S2] "
        "ngay sau luận điểm liên quan. Nếu context chưa đủ, nói rõ giới hạn."
    )
    user_prompt = (
        f"CONTEXT:\n{build_answer_context(sources)}\n\n"
        f"CÂU HỎI:\n{question}"
    )
    result = get_answer_llm().invoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ]
    )
    content = getattr(result, "content", result)
    if isinstance(content, list):
        content = " ".join(
            str(part.get("text", part)) if isinstance(part, dict) else str(part)
            for part in content
        )
    return clean_text(content)


def response_for_sample(
    response_mode: str,
    sample: dict[str, Any],
    question: str,
    reference: str,
    sources: list[dict[str, Any]],
    mode_label: str,
) -> str:
    ragas_response = clean_text((sample.get("ragas_fields") or {}).get("response") or "")
    if response_mode == "dataset":
        return ragas_response
    if response_mode == "reference":
        return reference
    if response_mode == "none":
        return ""
    if response_mode == "generate":
        return generate_answer_with_deepseek(question, sources, mode_label)
    raise ValueError(f"Unsupported RAGAS_RESPONSE_MODE: {response_mode}")

In [ ]:
NETWORK_RETRY_ATTEMPTS = max(1, env_int("RAGAS_NETWORK_RETRIES", 5) or 1)
NETWORK_RETRY_BASE_SECONDS = env_float("RAGAS_NETWORK_RETRY_BASE_SECONDS", 1.5)
NETWORK_RETRY_MAX_SECONDS = env_float("RAGAS_NETWORK_RETRY_MAX_SECONDS", 20.0)
MAX_CONSECUTIVE_NETWORK_ERRORS = env_int("RAGAS_MAX_CONSECUTIVE_NETWORK_ERRORS", 5)
TRANSIENT_NETWORK_MARKERS = (
    "getaddrinfo failed",
    "[errno 11001]",
    "[errno 11002]",
    "temporary failure in name resolution",
    "name or service not known",
    "failed to resolve",
    "connecterror",
    "connecttimeout",
    "readtimeout",
    "connectionreseterror",
    "connection aborted",
    "remoteprotocolerror",
    "server disconnected",
    "bad gateway",
    "gateway timeout",
    "too many requests",
    "service unavailable",
    "temporarily unavailable",
    " 429 ",
    " 500 ",
    " 502 ",
    " 503 ",
    " 504 ",
    "timed out",
)
PREPARE_ERRORS_BY_BACKEND: dict[str, list[dict[str, Any]]] = {}


def is_transient_network_error(exc: Exception) -> bool:
    text = f"{type(exc).__module__}.{type(exc).__name__}: {exc}".lower()
    status_code = getattr(exc, "status_code", None)
    if status_code in {408, 409, 425, 429, 500, 502, 503, 504, 520, 522, 524}:
        return True
    return any(marker in text for marker in TRANSIENT_NETWORK_MARKERS)


def reset_cached_backend(canonical_backend: str | None) -> None:
    if canonical_backend:
        store_cache.pop(canonical_backend, None)
        store_backend_errors.pop(canonical_backend, None)


def retry_transient(
    label: str,
    operation: Any,
    canonical_backend: str | None = None,
) -> Any:
    last_exc: Exception | None = None
    for attempt in range(1, NETWORK_RETRY_ATTEMPTS + 1):
        try:
            return operation()
        except Exception as exc:
            last_exc = exc
            if attempt >= NETWORK_RETRY_ATTEMPTS or not is_transient_network_error(exc):
                raise
            reset_cached_backend(canonical_backend)
            wait_seconds = min(
                NETWORK_RETRY_MAX_SECONDS,
                NETWORK_RETRY_BASE_SECONDS * (2 ** (attempt - 1)),
            )
            print(
                f"[retry] {label}: {type(exc).__name__}: {exc}; "
                f"retry in {wait_seconds:.1f}s",
                flush=True,
            )
            time.sleep(wait_seconds)
    raise last_exc or RuntimeError(f"{label} failed")


def retrieve_sources_for_sample(
    canonical_backend: str,
    question: str,
    top_k: int,
    backend_name: str,
    sample_id: str,
) -> list[dict[str, Any]]:
    return retry_transient(
        f"{backend_name} retrieve {sample_id}",
        lambda: get_store(canonical_backend).retrieve(question, top_k=top_k),
        canonical_backend=canonical_backend,
    )


def build_records_for_backend(
    backend: str,
    samples: list[dict[str, Any]],
    top_k: int,
    response_mode: str,
) -> list[dict[str, Any]]:
    backend_name = backend_slug(backend)
    canonical_backend = normalize_backend(backend)
    records: list[dict[str, Any]] = []
    prepare_errors: list[dict[str, Any]] = []
    PREPARE_ERRORS_BY_BACKEND[backend_name] = prepare_errors
    mode_label = (
        "Dataset"
        if canonical_backend == "dataset"
        else backend_mode_label(canonical_backend)
    )
    consecutive_network_errors = 0

    for index, sample in enumerate(samples, start=1):
        started = time.time()
        sample_id = sample.get("id") or f"sample_{index:06d}"
        question = question_from_sample(sample)
        reference = reference_from_sample(sample)
        if not question:
            raise ValueError(f"Sample {sample_id} has no question")

        try:
            if canonical_backend == "dataset":
                contexts = [
                    clean_text(value)
                    for value in (
                        (sample.get("ragas_fields") or {}).get("retrieved_contexts") or []
                    )
                    if clean_text(value)
                ]
                sources = []
            else:
                sources = retrieve_sources_for_sample(
                    canonical_backend,
                    question,
                    top_k,
                    backend_name,
                    sample_id,
                )
                contexts = [context_text_from_source(source) for source in sources]

            # RESPONSE_MODE=generate gọi DeepSeek local, không gọi API.
            response = response_for_sample(
                response_mode,
                sample,
                question,
                reference,
                sources,
                mode_label,
            )
            consecutive_network_errors = 0
        except Exception as exc:
            if not is_transient_network_error(exc):
                raise
            consecutive_network_errors += 1
            error_row = {
                "id": sample_id,
                "backend": backend_name,
                "canonical_backend": canonical_backend,
                "index": index,
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
            prepare_errors.append(error_row)
            if len(prepare_errors) <= 5 or len(prepare_errors) % 25 == 0:
                print(
                    f"[{backend_name}] skipped {sample_id}: "
                    f"{type(exc).__name__}: {exc}",
                    flush=True,
                )
            if (
                MAX_CONSECUTIVE_NETWORK_ERRORS
                and consecutive_network_errors >= MAX_CONSECUTIVE_NETWORK_ERRORS
            ):
                print(
                    f"[{backend_name}] stopped after "
                    f"{consecutive_network_errors} consecutive network errors.",
                    flush=True,
                )
                break
            continue

        exact_scores = exact_retrieval_scores(sources, expected_chunk_ids(sample))
        records.append(
            {
                "id": sample_id,
                "backend": backend_name,
                "canonical_backend": canonical_backend,
                "question_type": sample.get("question_type"),
                "difficulty": sample.get("difficulty"),
                "user_input": question,
                "reference": reference,
                "retrieved_contexts": contexts,
                "response": response,
                "expected_chunk_ids": expected_chunk_ids(sample),
                "retrieved_sources": [
                    compact_source(source) for source in sources
                ],
                "latency_seconds": round(time.time() - started, 4),
                **exact_scores,
            }
        )
        if index % 25 == 0:
            print(
                f"[{backend_name}] prepared {index}/{len(samples)} samples",
                flush=True,
            )
    return records


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_records_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    fields = [
        "id",
        "backend",
        "canonical_backend",
        "question_type",
        "difficulty",
        "exact_hit_at_k",
        "exact_hits",
        "exact_precision",
        "exact_recall",
        "exact_mrr",
        "latency_seconds",
        "user_input",
        "reference",
        "response",
    ]
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fields,
            extrasaction="ignore",
        )
        writer.writeheader()
        writer.writerows(rows)


def mean_or_none(values: list[Any]) -> float | None:
    numbers = []
    for value in values:
        if value is None or isinstance(value, bool):
            continue
        if isinstance(value, (int, float)) and not math.isnan(float(value)):
            numbers.append(float(value))
    return None if not numbers else sum(numbers) / len(numbers)


def exact_summary(records: list[dict[str, Any]]) -> dict[str, Any]:
    return {
        "sample_count": len(records),
        "exact_hit_rate": mean_or_none(
            [1.0 if row["exact_hit_at_k"] else 0.0 for row in records]
        ),
        "exact_precision": mean_or_none(
            [row["exact_precision"] for row in records]
        ),
        "exact_recall": mean_or_none(
            [row["exact_recall"] for row in records]
        ),
        "exact_mrr": mean_or_none([row["exact_mrr"] for row in records]),
        "avg_latency_seconds": mean_or_none(
            [row["latency_seconds"] for row in records]
        ),
    }


def ragas_rows(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [
        {
            "user_input": record["user_input"],
            "retrieved_contexts": record["retrieved_contexts"],
            "response": record["response"],
            "reference": record["reference"],
        }
        for record in records
    ]


def selected_metric_names(
    metric_names: list[str],
    response_mode: str | None = None,
) -> list[str]:
    mode = response_mode or RESPONSE_MODE
    names = list(dict.fromkeys(metric_names))
    if mode == "none":
        names = [name for name in names if name in CONTEXT_METRICS]
    missing = [
        name
        for name in names
        if name not in CONTEXT_METRICS | RESPONSE_METRICS
    ]
    if missing:
        raise ValueError(f"Unsupported metrics: {', '.join(missing)}")
    return names


def build_ragas_metrics(metric_names: list[str]) -> list[Any]:
    from ragas.metrics import (
        answer_correctness,
        answer_relevancy,
        context_precision,
        context_recall,
        faithfulness,
    )

    mapping = {
        "context_precision": context_precision,
        "context_recall": context_recall,
        "faithfulness": faithfulness,
        "answer_relevancy": answer_relevancy,
        "answer_correctness": answer_correctness,
    }
    return [copy.deepcopy(mapping[name]) for name in metric_names]


def build_ragas_embeddings(provider: str, model: str | None) -> Any:
    provider = provider.lower()
    if provider == "legal_hash":
        return LegalHashEmbeddings()
    if provider in LOCAL_EMBEDDING_PROVIDERS:
        try:
            from langchain_huggingface import HuggingFaceEmbeddings
        except ModuleNotFoundError:
            from langchain_community.embeddings import HuggingFaceEmbeddings

        # Model này cũng phải tồn tại/cached local vì HF_HUB_OFFLINE=1.
        return HuggingFaceEmbeddings(
            model_name=model or "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            model_kwargs={
                "device": "cuda:0",
                "local_files_only": True,
            },
            encode_kwargs={
                "batch_size": RAGAS_EMBEDDING_BATCH_SIZE,
                "normalize_embeddings": True,
            },
        )
    raise ValueError(
        "RAGAS_EMBEDDING_PROVIDER phải là legal_hash hoặc local huggingface"
    )


def ragas_summary_from_result(result: Any, metric_names: list[str]) -> dict[str, Any]:
    try:
        return dict(result)
    except Exception:
        pass
    try:
        frame = result.to_pandas()
    except Exception:
        return {}
    return {
        name: mean_or_none(frame[name].tolist())
        for name in metric_names
        if name in frame.columns
    }


def run_ragas_for_records(
    records: list[dict[str, Any]],
    metric_names: list[str],
) -> tuple[pd.DataFrame, dict[str, Any]]:
    from datasets import Dataset
    from ragas import evaluate

    metric_names = selected_metric_names(metric_names, RESPONSE_MODE)
    dataset = Dataset.from_list(ragas_rows(records))

    # RAGAS sẽ gọi DeepSeek local; không có provider/API remote.
    llm = build_chat_llm("deepseek_local", str(DEEPSEEK_MODEL_PATH))
    embeddings = build_ragas_embeddings(EMBEDDING_PROVIDER, EMBEDDING_MODEL)
    result = evaluate(
        dataset=dataset,
        metrics=build_ragas_metrics(metric_names),
        llm=llm,
        embeddings=embeddings,
        batch_size=RAGAS_BATCH_SIZE,
        raise_exceptions=RAISE_EXCEPTIONS,
        show_progress=True,
    )
    try:
        scores_df = result.to_pandas()
    except Exception:
        scores_df = pd.DataFrame(ragas_rows(records))

    scores_df.insert(0, "id", [row["id"] for row in records])
    scores_df.insert(1, "backend", [row["backend"] for row in records])
    scores_df.insert(2, "question_type", [row["question_type"] for row in records])
    scores_df.insert(3, "difficulty", [row["difficulty"] for row in records])
    return scores_df, ragas_summary_from_result(result, metric_names)

In [ ]:
def check_qdrant_api() -> tuple[str, str]:
    base = (os.getenv("QDRANT_URL") or "").rstrip("/")
    api_key = os.getenv("QDRANT_API_KEY") or ""
    if not base:
        return "not_configured", "QDRANT_URL is unset"
    try:
        response = requests.get(
            f"{base}/collections",
            headers={"api-key": api_key} if api_key else {},
            timeout=8,
        )
    except Exception as exc:
        return "connection_error", f"{type(exc).__name__}: {exc}"
    if response.status_code == 200:
        return "ok", "Qdrant REST API reachable"
    if response.status_code in {401, 403}:
        return "auth_error", f"HTTP {response.status_code}; check QDRANT_API_KEY"
    return "http_error", f"HTTP {response.status_code}; {response.text[:120]}"


def check_neo4j_socket() -> tuple[str, str]:
    uri = os.getenv("NEO4J_URI") or ""
    if not uri:
        return "not_configured", "NEO4J_URI is unset"
    parsed = urlparse(uri)
    host = parsed.hostname or "127.0.0.1"
    port = parsed.port or 7687
    try:
        with socket.create_connection((host, port), timeout=4):
            return "ok", f"Bolt socket reachable at {host}:{port}"
    except Exception as exc:
        return "connection_error", f"{type(exc).__name__}: {exc}"


def backend_env_requirements(backend: str) -> dict[str, Any]:
    canonical = normalize_backend(backend)
    row = {"backend": backend, "canonical_backend": canonical}
    if canonical in {"qdrant", "hybrid"}:
        status, detail = check_qdrant_api()
        row.update(
            {
                "qdrant_url": os.getenv("QDRANT_URL", "unset"),
                "qdrant_collection": os.getenv("QDRANT_COLLECTION", "unset"),
                "qdrant_status": status,
                "qdrant_detail": detail,
            }
        )
    if canonical in {"neo4j", "hybrid"}:
        status, detail = check_neo4j_socket()
        row.update(
            {
                "neo4j_uri": os.getenv("NEO4J_URI", "unset"),
                "neo4j_user": os.getenv("NEO4J_USER", "unset"),
                "neo4j_database": os.getenv("NEO4J_DATABASE", "unset"),
                "neo4j_status": status,
                "neo4j_detail": detail,
            }
        )
    return row


backend_config_df = pd.DataFrame(
    [backend_env_requirements(backend) for backend in BACKENDS]
)
display(backend_config_df)

samples = load_eval_samples(DATASET_PATH, offset=OFFSET, limit=LIMIT)
print(f"Loaded {len(samples)} samples from {DATASET_PATH}")
print(f"Output directory: {RUN_DIR}")

prepared_by_backend: dict[str, list[dict[str, Any]]] = {}
summary_rows: list[dict[str, Any]] = []

if RESPONSE_MODE == "none":
    metric_names = [name for name in METRIC_NAMES if name in CONTEXT_METRICS]
else:
    metric_names = list(METRIC_NAMES)

for backend in BACKENDS:
    backend_name = backend_slug(backend)
    print(f"\nPreparing backend: {backend_name}")
    try:
        records = build_records_for_backend(
            backend,
            samples,
            TOP_K,
            RESPONSE_MODE,
        )
    except Exception as exc:
        print(f"SKIP {backend_name}: {type(exc).__name__}: {exc}")
        continue

    prepared_by_backend[backend_name] = records
    write_jsonl(RUN_DIR / f"{backend_name}_prepared.jsonl", records)
    write_records_csv(RUN_DIR / f"{backend_name}_prepared.csv", records)
    prepare_errors = PREPARE_ERRORS_BY_BACKEND.get(backend_name, [])
    if prepare_errors:
        write_jsonl(
            RUN_DIR / f"{backend_name}_prepare_errors.jsonl",
            prepare_errors,
        )

    summary_rows.append(
        {
            "backend": backend_name,
            "canonical_backend": (
                records[0]["canonical_backend"]
                if records
                else normalize_backend(backend)
            ),
            "response_mode": RESPONSE_MODE,
            "top_k": TOP_K,
            "prepare_error_count": len(prepare_errors),
            **exact_summary(records),
        }
    )

comparison_df = pd.DataFrame(summary_rows)
comparison_df.to_csv(
    RUN_DIR / "comparison_exact.csv",
    index=False,
    encoding="utf-8-sig",
)
comparison_df

In [ ]:
ragas_summary_rows = []
ragas_error_rows = []

if RUN_RAGAS:
    metric_names = selected_metric_names(metric_names)
    print("RAGAS metrics:", metric_names)
    print("RAGAS batch size:", RAGAS_BATCH_SIZE)
    for backend_name, records in prepared_by_backend.items():
        base_summary = next(
            (
                row
                for row in summary_rows
                if row["backend"] == backend_name
            ),
            {"backend": backend_name},
        )
        if not records:
            error_row = {
                **base_summary,
                "ragas_error_type": "NoPreparedRecords",
                "ragas_error": "no prepared records",
            }
            ragas_summary_rows.append(error_row)
            continue

        print(f"\nRunning RAGAS for {backend_name} ({len(records)} samples)")
        try:
            scores_df, ragas_summary = run_ragas_for_records(
                records,
                metric_names,
            )
        except Exception as exc:
            error_row = {
                **base_summary,
                "ragas_error_type": type(exc).__name__,
                "ragas_error": str(exc),
            }
            ragas_summary_rows.append(error_row)
            ragas_error_rows.append(error_row)
            print(
                f"RAGAS failed for {backend_name}: "
                f"{type(exc).__name__}: {exc}"
            )
            continue

        scores_df.to_csv(
            RUN_DIR / f"{backend_name}_ragas_scores.csv",
            index=False,
            encoding="utf-8-sig",
        )
        ragas_summary_rows.append({**base_summary, **ragas_summary})

    if ragas_error_rows:
        ragas_errors_df = pd.DataFrame(ragas_error_rows)
        ragas_errors_df.to_csv(
            RUN_DIR / "ragas_errors.csv",
            index=False,
            encoding="utf-8-sig",
        )
        (
            RUN_DIR / "ragas_errors.json"
        ).write_text(
            ragas_errors_df.to_json(
                orient="records",
                force_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
    final_comparison_df = pd.DataFrame(ragas_summary_rows)
else:
    final_comparison_df = pd.DataFrame(summary_rows)

final_comparison_df.to_csv(
    RUN_DIR / "comparison.csv",
    index=False,
    encoding="utf-8-sig",
)
(
    RUN_DIR / "comparison.json"
).write_text(
    final_comparison_df.to_json(
        orient="records",
        force_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
final_comparison_df

In [ ]:
for backend_name, records in prepared_by_backend.items():
    misses = [row for row in records if not row["exact_hit_at_k"]]
    print(f"{backend_name}: {len(misses)} exact retrieval misses")
    if misses:
        sample = misses[0]
        print("id:", sample["id"])
        print("question:", sample["user_input"])
        print("expected_chunk_ids:", sample["expected_chunk_ids"])
        print(
            "retrieved_chunk_ids:",
            [source["chunk_id"] for source in sample["retrieved_sources"]],
        )